# Real data: sampling every timing parameter on J1640+2224

Apply the workflow from notebooks **01** and **02** to real EPTA DR2
J1640+2224: sample **all** timing fitpars jointly with red noise (full-basis /
“decentering” style analysis).

You will:

1. Build a full-basis context and inspect each axis’s chart.
2. Read the transport-center report **chart-type-first** (large `|center_z|` is a
   boundary warning only on PIT charts).
3. See why declaring spin/astrometry `identically_linear` improves off-mode
   geometry — and why a real binary pulsar still may not fully “pass”
   certification (nonlinear binary parameters + timing↔noise curvature).
4. Refine the expansion, write a geometry report, run a short NUTS chain
   (diagnose chains separately), and compare pivot vs 1/yr red-noise amplitude.

**Setup:** MetaPulsar is required today for the `TimingPulsar` contract (see
notebook **01**). The EPTA DR2 J1640 par/tim are expected under a MetaPulsar
checkout at `data/ipta-dr2/.../J1640+2224` (this cell walks parents to find
that tree). Warmup/samples are small for pedagogy — scale up for science.
Read **01**–**02** first if the vocabulary is new.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
from pathlib import Path

import jax
import numpy as np
import matplotlib.pyplot as plt

import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel, box_hyper_probe_points, certify_joint_geometry, transport_center_report,
    write_geometry_report, read_geometry_report, refine_timing_expansion,
)
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")

# Real IPTA-DR2 J1640+2224 (EPTA v2.2). Find the repo root from the CWD.
_here = Path.cwd()
_repo = next(p for p in (_here, *_here.parents) if (p / "data" / "ipta-dr2").is_dir())
PAR = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224.par"
TIM = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224_all.tim"
assert PAR.exists(), f"J1640 par not found: {PAR}"

mp = create_metapulsar(
    {"epta": [{"par": str(PAR), "tim": str(TIM), "timing_package": "tempo2"}]},
    use_pulse_numbers="no",
)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="J1640 WN reference")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
print(f"{mp.name}: {len(mp.toas)} TOAs, {len(mp.fitpars)} fitpars")
print("fitpars:", list(mp.fitpars))


## 1. Full-basis context and coordinate charts

`inference="all"` means every timing fitpar is sampled (nothing analytically
marginalized). Inspect `chart_summary`: which axes are `affine_normal` (Gaussian
δ prior, globally affine) vs `prior_pit` (bounded/uniform prior → nonlinear
PIT, exact throughout the prior support)? Which does the registry already mark
identically linear? Geometry concerns below are about off-mode *likelihood*
curvature under those charts — not about the charts themselves being only
locally valid.


In [ ]:
ctx = NonLinearTimingModel(
    engines="jug", inference="all", name="timing"
).for_pulsar(mp)

print(f"{'axis':12s} {'disposition':12s} {'chart':14s} {'lin?':5s} prior")
for d in ctx.chart_summary():
    print(f"{d['name']:12s} {d['disposition']:12s} {d['chart']:14s} "
          f"{str(d['identically_linear']):5s} {d['prior_family']}")
print("\nidentically linear:", ctx.identically_linear)
print("resolved tempo2_native:", ctx.model.resolved_tempo2_native)


## 2. Is the conditioned center inside each chart’s domain?

For each axis: where does the red-noise-conditioned center sit in z, and is that
still interior to the chart’s domain (the prior support in z)? Always read the
**chart type first**:

- `affine_normal` — interior for any finite center (Gaussian mean shift).
- `prior_pit` — finite boundary from the prior support; only here is large
  `|center_z|` a saturation warning (the chart map itself is still exact in the
  interior).

White noise is fixed for a fast build; red-noise hypers are freed later for NUTS.


In [ ]:
psl = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx.discovery_signals(joint=True),
])
jm = N.joint_model(psl, ctx, reference_noise=reference, fixed=nd)

for a in transport_center_report(ctx, jm.transport, center):
    kind = "affine (no boundary)" if a.chart == "affine_normal" else \
        ("PIT interior" if a.interior else "PIT SATURATED")
    print(f"{a.name:12s} {a.chart:14s} center_z={a.center_z:+7.3f}  {kind}")


## 3. Declare linear axes — then see what remains hard

As in notebook **02**, spindown is not identically linear in the residual
(`r ∼ ΔΦ/F0`), but for PTA MSPs it is *numerically* extremely close — so we
declare spin and proper-motion/parallax `identically_linear` (unioned with the
auto-derived set — an explicit list *replaces* the auto set). That flips their
default wide uniform PIT charts to `affine_normal`. Expect the Hessian
eigenvalues to move toward 1 and the residual RMS to drop sharply.

On real J1640 the report often still does **not** fully pass afterward: binary
parameters (`PB`, `A1`, …) are genuinely nonlinear, and a white-noise-only
reference cannot precondition timing↔red-noise cross-curvature. Treat that as
information about the model, not a bar to force under.


In [ ]:
bounds = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
          f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
probes = box_hyper_probe_points(center, bounds)[:1]
report_default = certify_joint_geometry(jm, ctx, hyper_points=probes)

linear_axes = {"F0", "F1", "RAJ", "DECJ", "PMRA", "PMDEC", "PX"} & set(mp.fitpars)
declared = sorted(set(ctx.identically_linear) | linear_axes)
ctx_lin = NonLinearTimingModel(
    engines="jug", inference="all",
    identically_linear=declared, name="timing",
).for_pulsar(mp)
psl_lin = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=True)])
jm_lin = N.joint_model(psl_lin, ctx_lin, reference_noise=reference, fixed=nd)
report_lin = certify_joint_geometry(jm_lin, ctx_lin, hyper_points=probes)


def _row(lbl, r):
    return (f"  {lbl:22s} rms={r.max_residual_remainder_rms:11.4g}"
            f"  H_eig=[{r.xi_hessian_eigen_min:9.3g}, {r.xi_hessian_eigen_max:10.4g}]"
            f"  xi_grad={r.max_xi_gradient_inf_norm:10.4g}"
            f"  passed={r.passed}")


print("declared linear:", sorted(linear_axes))
print(_row("wide uniform PIT", report_default))
print(_row("identically_linear", report_lin))
print(f"  -> H eigenvalue max: {report_default.xi_hessian_eigen_max:.3g} "
      f"-> {report_lin.xi_hessian_eigen_max:.3g}")


## 4. Refine the expansion at the WN + hyper point

Same idea as notebook **02**: maximize the marginal timing objective at the
fixed white-noise + red-noise hyper point, re-linearize there. Continue with the
identically-linear context (`ctx_lin`).


In [ ]:
psl_marg = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=False)])
objective = N.conditional_timing_potential(psl_marg, ctx_lin, fixed={**nd, **center})
refined = refine_timing_expansion(ctx_lin, negative_log_target_z=objective)
ctx_run = refined.context
z0 = np.asarray(ctx_lin.linearization.sampled_z_expansion)
z1 = np.asarray(ctx_run.linearization.sampled_z_expansion)
print("refine converged:", refined.converged, " |d z_e|:", float(np.linalg.norm(z1 - z0)))


## 5. Certify and save a geometry report

Probe a few hyper points, print the summary, and write a standalone JSON+NPZ
geometry product (digest-verified on reload). This is independent of the
sampling run — useful to archive next to chains.


In [ ]:
psl_run = ds.PulsarLikelihood([
    mp.residuals, ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_run.discovery_signals(joint=True)])
jm_run = N.joint_model(psl_run, ctx_run, reference_noise=reference, fixed=nd)

report = certify_joint_geometry(jm_run, ctx_run, hyper_points=box_hyper_probe_points(center, bounds)[:3])
print("passed:", report.passed, "| failures:", len(report.failures),
      "| rms:", round(report.max_residual_remainder_rms, 4))

outdir = Path(os.environ.get("TMPDIR", "/tmp")) / "j1640_decentering"
outdir.mkdir(parents=True, exist_ok=True)
jp, npzp = write_geometry_report(report, outdir / "geometry", overwrite=True)
print("wrote", jp.name, "+", npzp.name,
      "| roundtrip:", read_geometry_report(outdir / "geometry").model_fingerprint == report.model_fingerprint)


## 6. Short NUTS run — diagnose chains separately

Free the red-noise hyperparameters. `N.nuts` with `dense_mass="auto"` adapts a
dense mass block on those hypers while leaving the timing latent on an identity
mass. Plot **each chain** before any pooled corner — a frozen chain next to a
moving one is the usual failure signature. Warmup/samples are small here.


In [ ]:
rn_priors = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
             f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
jm_free = N.joint_model(psl_run, ctx_run, reference_noise=reference,
                        priors=rn_priors, fixed=nd)
print("hyper sites (block mass):", jm_free.hyper_sites)

mcmc = N.nuts(jm_free, ctx_run, num_warmup=400, num_samples=400, num_chains=2,
              chain_method="sequential", progress_bar=True)
mcmc.run(jax.random.PRNGKey(0), extra_fields=N.NUTS_EXTRA_FIELDS)

diag = N.chain_diagnostics(mcmc, max_tree_depth=10)
print("tree-depth saturation:", round(diag['tree_depth_saturation_fraction'], 3),
      "| mean accept:", round(float(diag['accept_prob'].mean()), 3))

# Plot each chain's red-noise amplitude trace SEPARATELY (never pool first).
amp = f"{mp.name}_rednoise_log10_A"
sam = mcmc.get_samples(group_by_chain=True)[amp]
fig, axes = plt.subplots(1, sam.shape[0], figsize=(9, 3), sharey=True)
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.plot(np.asarray(sam[i]), lw=0.6)
    ax.set_title(f"chain {i}")
    ax.set_xlabel("draw")
axes[0].set_ylabel("log10_A")
fig.tight_layout()


## 7. Pivot vs 1/yr red-noise amplitude

Same tip as notebook **02**: amplitude at a sensitivity-weighted pivot
frequency vs at 1/yr. Decode pivot draws back to 1/yr for comparison.


In [ ]:
f, df, fmat = ds.fourierbasis(mp, 10)
w = ds.fourier_sensitivity_weights(fmat, dst.reference_noise(mp))
f_pivot = ds.sensitivity_weighted_pivot_frequency(np.asarray(f)[0::2], w)
print(f"pivot frequency: {f_pivot:.3e} Hz   (1/yr = {ds.const.fyr:.3e} Hz)")

param = ds.PowerLawParameterization(slope_pivot_frequency=f_pivot)
gamma = 3.5
for a_pivot in (-14.0, -14.5):
    a_ref = ds.reference_log10_amplitude(a_pivot, gamma, f_pivot=f_pivot, parameterization=param)
    print(f"  log10_A_pivot={a_pivot:.2f} (at f_pivot)  ==  log10_A={a_ref:.4f} (at 1/yr)")


## Summary

- Always read the **chart type** before interpreting `|center_z|`: only PIT
  charts have a finite prior-support boundary (the chart map is exact in the
  interior; large `|center_z|` is a domain/saturation warning).
- Declaring near-linear PTA axes (spin, astrometry) `identically_linear` is a
  modeling fix for off-mode geometry, not a NUTS tuning knob.
- On real J1640, expect residual certification failures from binary nonlinearities
  and timing↔red-noise curvature even after that fix — use them to guide the next
  modeling step, not to loosen thresholds.
